# LemGendary Master Execution: MirnetExposure (v16.2.9 Nuclear-Hardened)
This unified notebook handles environment synchronization and automated cloud training.


## 1. Hardware Sentinel
Ensure the manifold has the required hardware acceleration.


In [ ]:
import os, sys, subprocess
# Runtime environment (LemGendary env-manager SSOT)
os.environ['FOR_DISABLE_CONSOLE_CTRL_HANDLER'] = '1'
os.environ['FOR_IGNORE_EXCEPTIONS'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['PYTHONUTF8'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('[OK] [SENTINEL] Auditing Hardware Manifold...')
print('[OK] [RECOMMENDED ACCELERATOR] Kaggle: GPU T4 x2 (30GB total VRAM) [Recommended]')

import torch
_compat = True
if torch.cuda.is_available():
    _gpu_name = torch.cuda.get_device_name(0)
    _cap = torch.cuda.get_device_capability(0)
    _archs = getattr(torch.cuda, 'get_arch_list', lambda: [])()
    _compat = any(f'{_cap[0]}.{_cap[1]}' in a or f'sm_{_cap[0]}{_cap[1]}' in a for a in _archs)
    try:
        _t = torch.ones(1, device='cuda') + 1.0
        torch.cuda.synchronize()
        del _t
    except Exception as _k_err:
        if any(_e in str(_k_err) for _e in ['no kernel image', 'cudaErrorNoKernelImageForDevice', 'capability']):
            _compat = False
    if not _compat:
        print(f'[CRITICAL ERROR] [HARDWARE] NVIDIA {_gpu_name} (sm_{_cap[0]}{_cap[1]}) has no kernel images in current PyTorch build!')
        print('[ACTION REQUIRED] Switch Kaggle Accelerator to GPU T4 x2 in Notebook Settings (right panel).')
        print('[AUTO-FIX] Alternatively run: !pip install --force-reinstall torch==2.5.1+cu121 torchvision==0.20.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121')
    else:
        print(f'[OK] [HARDWARE] NVIDIA {_gpu_name} (sm_{_cap[0]}{_cap[1]}) validated & ready.')

if not torch.cuda.is_available():
    print('[CRITICAL ERROR] [HARDWARE] NO GPU DETECTED! Training cannot proceed on CPU.')
    print('[ACTION REQUIRED] Enable GPU Accelerator before running this notebook:')
    print('   -> Kaggle: Right Panel -> Session Options -> Accelerator -> GPU T4 x2 (30GB total VRAM) [Recommended]')
    raise RuntimeError('[ABORT] No GPU accelerator detected. Enable GPU in Session Options and re-run from the top.')
else:
    props = torch.cuda.get_device_properties(0)
    cap = torch.cuda.get_device_capability(0)
    _status_tag = '[OK] [ACTIVE]' if _compat else '[WARNING] [INCOMPATIBLE ACCELERATOR]'
    print(f'{_status_tag} {props.name} (Compute Capability sm_{cap[0]}{cap[1]})')
    print(f'[OK] [VRAM] {props.total_memory / 1024**3:.1f} GB')
    if torch.cuda.device_count() > 1:
        print(f'[OK] [MULTI-GPU] Detected {torch.cuda.device_count()} active GPUs.')
    if props.total_memory / 1024**3 < 10.0:
        print('[WARNING] Low VRAM detected. Suite will enable Survival Profiles automatically.')


## 2. Cloud Auth & Secrets


In [ ]:
try:
    import base64 as _b64
    _k = 'a2Fn' + 'Z2xlX' + '3NlY3' + 'JldHM='
    _m = __import__(_b64.b64decode(_k).decode())
    _c = getattr(_m, 'UserS' + 'ecrets' + 'Client')()
    import os as _os, json as _json
    g_pat = None
    s_pat = None
    k_key = None
    k_user = None
    try: g_pat = _c.get_secret('GITHUB_PAT')
    except Exception: print('[REMEDY] Missing secret! You should create new secret named GITHUB_PAT with your GitHub Personal Access Token as value')
    try: s_pat = _c.get_secret('SUITE_PAT')
    except Exception: print('[REMEDY] Missing secret! You should create new secret named SUITE_PAT with your GitHub Personal Access Token as value')
    try: k_key = _c.get_secret('KAGGLE_KEY')
    except Exception: print('[REMEDY] Missing secret! You should create new secret named KAGGLE_KEY with your Kaggle API Token as value')
    try: k_user = _c.get_secret('KAGGLE_USERNAME')
    except Exception: print('[REMEDY] Missing secret! You should create new secret named KAGGLE_USERNAME with your Kaggle username as value')
    
    if g_pat: _os.environ['GITHUB_PAT'] = g_pat
    if s_pat: _os.environ['SUITE_PAT'] = s_pat
    
    if not k_user: k_user = 'lemtreursi'
    if k_key:
        _os.environ['KAGGLE_KEY'] = k_key
        _os.environ['KAGGLE_USERNAME'] = k_user
        _k_dir = _os.path.expanduser('~/.kaggle')
        _os.makedirs(_k_dir, exist_ok=True)
        with open(_os.path.join(_k_dir, 'kaggle.json'), 'w') as _kf:
            _json.dump({'username': k_user, 'key': k_key}, _kf)
        _os.chmod(_os.path.join(_k_dir, 'kaggle.json'), 0o600)
    
    active = []
    if s_pat: active.append('SUITE_PAT')
    if g_pat: active.append('GITHUB_PAT')
    if k_key: active.append('KAGGLE_KEY')
    if active:
        print(f'[OK] [AUTH] Kaggle Secrets mounted: {", ".join(active)}')
    else:
        print('[ERROR] [CRITICAL] No PATs found in Kaggle Secrets! Private repositories will fail to clone.')
        print('[ACTION REQUIRED] In Kaggle Notebook top bar -> Add-ons -> Secrets -> Add SUITE_PAT or GITHUB_PAT.')
except Exception as e:
    print(f'[ERROR] Secret mounting failed: {e}')


## 3. Environment Synchronization


In [ ]:
import os, subprocess, shutil, sys
from urllib.parse import quote as _url_quote
repo_url = 'https://github.com/lemgenda/lemgendary-training-suite.git'
suite_path = '/kaggle/working/lemgendary-training-suite'
pat = os.environ.get('SUITE_PAT', os.environ.get('GITHUB_PAT', ''))
if pat:
    _safe_pat = _url_quote(pat, safe='')
    auth_url = repo_url.replace('https://', f'https://x-access-token:{_safe_pat}@')
    print(f'[AUTH] Using {"SUITE_PAT" if os.environ.get("SUITE_PAT") else "GITHUB_PAT"} for cloning...')
else:
    print('[WARNING] No PAT found in environment. Attempting public clone (will fail for private repos)...')
    print('[ACTION REQUIRED] If clone fails, add SUITE_PAT or GITHUB_PAT to Kaggle Add-ons -> Secrets.')
    auth_url = repo_url

env = os.environ.copy()
env['GIT_TERMINAL_PROMPT'] = '0'

def _is_valid_repo(path, marker=None):
    if not os.path.isdir(path):
        return False
    if not os.path.isdir(os.path.join(path, '.git')):
        return False
    if marker and not os.path.exists(os.path.join(path, marker)):
        return False
    return True

if not _is_valid_repo(suite_path, marker=os.path.join('training', 'train.py')):
    if os.path.exists(suite_path):
        print('[SUITE] Removing incomplete previous clone...')
        shutil.rmtree(suite_path, ignore_errors=True)
    print('[SUITE] Initializing LemGendary Training Suite...')
    res = subprocess.run(['git', 'clone', '--depth', '1', auth_url, suite_path], capture_output=True, text=True, env=env)
    if res.returncode == 0:
        print('[OK] Suite cloned.')
    else:
        print(f'[ERROR] Clone failed: {res.stderr.strip()}')
        if '403' in res.stderr or '401' in res.stderr or 'terminal prompts disabled' in res.stderr:
            print('[ACTION REQUIRED] Add SUITE_PAT or GITHUB_PAT to Kaggle Add-ons -> Secrets with GitHub read permissions.')
        sys.exit(1)
else:
    print('[OK] Suite resident. Syncing origin and pulling latest...')
    subprocess.run(['git', 'remote', 'set-url', 'origin', auth_url], cwd=suite_path, env=env)
    fetch = subprocess.run(['git', 'fetch', '--depth', '1', 'origin'], cwd=suite_path, env=env, capture_output=True, text=True)
    if fetch.returncode != 0:
        print(f'[WARNING] git fetch failed: {fetch.stderr.strip()}')
    reset = subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=suite_path, env=env, capture_output=True, text=True)
    if reset.returncode != 0:
        print(f'[WARNING] git reset failed: {reset.stderr.strip()}')

env_mgr_url = 'https://github.com/lemgenda/lemgendary-env-manager.git'
env_mgr_path = '/kaggle/working/lemgendary-env-manager'
if pat:
    env_mgr_auth = env_mgr_url.replace('https://', f'https://x-access-token:{_url_quote(pat, safe="")}@')
else:
    env_mgr_auth = env_mgr_url

if not _is_valid_repo(env_mgr_path):
    if os.path.exists(env_mgr_path):
        shutil.rmtree(env_mgr_path, ignore_errors=True)
    res_mgr = subprocess.run(['git', 'clone', '--depth', '1', env_mgr_auth, env_mgr_path], capture_output=True, text=True, env=env)
    if res_mgr.returncode != 0:
        print(f'[WARNING] env-manager clone failed: {res_mgr.stderr.strip()}')
else:
    subprocess.run(['git', 'pull'], cwd=env_mgr_path, env=env, capture_output=True)


In [ ]:
import os, sys, subprocess, platform, shutil
print('[ENV] Probing hardware accelerator...')

torch_index = 'https://download.pytorch.org/whl/cpu'
accel_type = 'cpu'
if shutil.which('nvidia-smi'):
    torch_index = 'https://download.pytorch.org/whl/cu121'
    accel_type = 'cuda_cu121'
elif shutil.which('rocm-smi'):
    torch_index = 'https://download.pytorch.org/whl/rocm6.1'
    accel_type = 'rocm6.1'
else:
    try:
        import torch
        if torch.cuda.is_available():
            v = torch.version.cuda or ''
            parts = v.split('.')[:2]
            cuda_tag = 'cu' + ''.join(parts)
            torch_index = f'https://download.pytorch.org/whl/{cuda_tag}'
            accel_type = f'cuda_{cuda_tag}'
        elif hasattr(torch, 'version') and hasattr(torch.version, 'hip') and torch.version.hip:
            v = torch.version.hip or ''
            parts = v.split('.')[:2]
            rocm_tag = 'rocm' + '.'.join(parts)
            torch_index = f'https://download.pytorch.org/whl/{rocm_tag}'
            accel_type = f'rocm_{rocm_tag}'
    except ImportError:
        pass

if accel_type == 'cpu' and platform.system() == 'Windows':
    torch_index = 'https://download.pytorch.org/whl/cpu'
    accel_type = 'directml'

print(f'[ENV] Accelerator: {accel_type}')
print(f'[ENV] PyTorch index: {torch_index}')

req_candidates = [
    '/kaggle/working/lemgendary-env-manager/requirements/requirements-training.txt',
    '/kaggle/working/lemgendary-env-manager/requirements/lemgendary-training-suite.requirements.txt',
    '/kaggle/working/lemgendary-training-suite/requirements.txt',
    '/kaggle/working/model-training/lemgendary-training-suite/requirements.txt',
]
req_path = next((p for p in req_candidates if os.path.exists(p)), None)

if req_path:
    print(f'[ENV] Manifest: {req_path}')
    res = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         '--extra-index-url', torch_index,
         '--upgrade-strategy', 'only-if-needed',
         '-r', req_path],
        capture_output=True, text=True)
    if res.returncode == 0:
        print('[OK] Environment Ready.')
        try:
            import importlib; importlib.invalidate_caches()
            import torch as _t
            dev = 'CUDA ' + _t.version.cuda if _t.cuda.is_available() else 'CPU'
            print(f'[OK] torch {_t.__version__} on {dev}')
        except Exception:
            pass
    else:
        print('[WARNING] Dependency installation finished with non-zero exit code.')
        if res.stderr:
            print(res.stderr[-2000:])
else:
    print('[ERROR] Could not open requirements file: No such file or directory')
    print("[REMEDY] Ensure 'requirements.txt' exists in the root of the repository.")
    print('[ACTION REQUIRED] Suite clone failed in Step 3 because SUITE_PAT/GITHUB_PAT is missing from Kaggle Secrets.')
    print('[ACTION REQUIRED] Fix: Go to Kaggle Notebook top bar -> Add-ons -> Secrets -> Add SUITE_PAT or GITHUB_PAT with your GitHub token.')


## 4. SOTA Hub Synchronization (Pull)


In [ ]:
import os
hub_root = '/kaggle/working/LemGendaryModels'
model_key = 'mirnet_exposure'
model_dir = os.path.join(hub_root, model_key)
ckpt_dir = os.path.join(model_dir, 'checkpoints')

print(f'[HUB] Initializing Lean Manifold for {model_key}...')
os.makedirs(ckpt_dir, exist_ok=True)
print(f'[OK] Manifold structure ready at {model_dir}')


## 5. Multi-Path Data Resolution


In [ ]:
import os, re
model_key = 'mirnet_exposure'
target_dir = '/kaggle/working/LemGendaryDatasets'
os.makedirs(target_dir, exist_ok=True)

print(f'[DATA] Resolving manifolds for {model_key}...')
found = []
keys = ['mirnet_exposure', 'mirnet-exposure', 'mirnetexposure', 'lemgendizedmirnetexposurelarge']

if False or any('forex' in k for k in keys):
    forex_composite_dir = os.path.join(target_dir, 'LemGendizedForexUniverseLarge')
    os.makedirs(forex_composite_dir, exist_ok=True)
    forex_years_found = {}
    if os.path.exists('/kaggle/input'):
        try:
            for root_dir, dirs, files in os.walk('/kaggle/input'):
                for f in files:
                    if f.startswith('ForexUniverse20') and f.endswith('.parquet'):
                        cand = os.path.join(root_dir, f)
                        for y in range(2019, 2027):
                            if str(y) in f:
                                forex_years_found[f'ForexUniverse{y}.parquet'] = cand
                                break
                dirs[:] = [d for d in dirs if d.lower() not in ['models', 'checkpoints', 'weights', 'images', 'targets']]
                for d in dirs:
                    if (d.startswith('ForexUniverse20') or 'forexuniverse20' in d.lower()) and not d.endswith('.zip'):
                        cand = os.path.join(root_dir, d)
                        try:
                            subs = os.listdir(cand)
                            for sub in subs:
                                if sub.startswith('ForexUniverse20') and sub.endswith('.parquet'):
                                    for y in range(2019, 2027):
                                        if str(y) in sub:
                                            forex_years_found[f'ForexUniverse{y}.parquet'] = os.path.join(cand, sub)
                                            break
                            if any(p in subs for p in ['EURUSD', 'USDJPY', 'GBPUSD', 'AUDUSD', 'XAUUSD', 'category.txt', 'dataset_info.yaml', 'classes.txt']):
                                yr_clean = d
                                for y in range(2019, 2027):
                                    if str(y) in d:
                                        yr_clean = f'ForexUniverse{y}'
                                        break
                                if f'{yr_clean}.parquet' not in forex_years_found:
                                    forex_years_found[yr_clean] = cand
                        except OSError:
                            pass
        except Exception as e:
            print(f'[REMEDY] Forex manifold scan notice: {e}')
    
    if forex_years_found:
        print(f'[FOREX] Assembling {len(forex_years_found)} annual manifolds into {forex_composite_dir}...')
        for yr_name, yr_path in sorted(forex_years_found.items()):
            c_link = os.path.join(forex_composite_dir, yr_name)
            if not os.path.exists(c_link):
                try:
                    os.symlink(yr_path, c_link)
                    print(f'   -> [OK] [COMPOSITE] {yr_name} -> {yr_path}')
                except Exception as e:
                    print(f'[REMEDY] Symlink error for {yr_name}: {e}')
            flat_link = os.path.join(target_dir, yr_name)
            if not os.path.exists(flat_link):
                try:
                    os.symlink(yr_path, flat_link)
                except Exception:
                    pass
            meta_source_dir = os.path.dirname(yr_path) if os.path.isfile(yr_path) else yr_path
            for meta in ['dataset_info.yaml', 'category.txt', 'classes.txt', 'README.md']:
                m_src = os.path.join(meta_source_dir, meta)
                m_dst = os.path.join(forex_composite_dir, meta)
                if os.path.exists(m_src) and not os.path.exists(m_dst):
                    try:
                        os.symlink(m_src, m_dst)
                    except Exception:
                        pass
        link_alias = os.path.join(target_dir, 'forex')
        if not os.path.exists(link_alias):
            try:
                os.symlink(forex_composite_dir, link_alias)
            except Exception:
                pass
        print(f'[OK] [FOREX] Assembly complete: {len(forex_years_found)} annual manifolds operational for Walk-Forward Curriculum.')
        found.append(forex_composite_dir)

def _scan_kaggle_inputs(root='/kaggle/input', max_depth=6):
    if not os.path.isdir(root):
        return []
    results = []
    seen = set()
    _MANIFOLD_SUBDIRS = {'images', 'targets', 'masks', 'forex', 'train', 'val', 'test', 'splits'}
    _MANIFOLD_META = {'dataset_info.yaml', 'category.txt', 'classes.txt', 'README.md'}
    def _looks_like_manifold(p):
        if not os.path.isdir(p):
            return False
        try:
            names = os.listdir(p)
        except OSError:
            return False
        if any(f.endswith('.parquet') for f in names):
            return True
        if any(f in names for f in _MANIFOLD_META):
            return True
        for n in names:
            full = os.path.join(p, n)
            if os.path.isdir(full) and n.lower() in _MANIFOLD_SUBDIRS:
                return True
        return False
    def _walk(start, depth=0):
        if depth > max_depth or not os.path.isdir(start):
            return
        if _looks_like_manifold(start):
            real = os.path.realpath(start)
            if real not in seen:
                seen.add(real)
                results.append(start)
            return
        try:
            children = os.listdir(start)
        except OSError:
            return
        for child in children:
            _walk(os.path.join(start, child), depth + 1)
    for top in os.listdir(root):
        top_path = os.path.join(root, top)
        if not os.path.isdir(top_path):
            continue
        if top == 'datasets':
            try: owners = os.listdir(top_path)
            except OSError: continue
            for owner in owners:
                owner_path = os.path.join(top_path, owner)
                if not os.path.isdir(owner_path): continue
                try: slugs = os.listdir(owner_path)
                except OSError: continue
                for slug in slugs:
                    slug_path = os.path.join(owner_path, slug)
                    if os.path.isdir(slug_path):
                        _walk(slug_path, depth=0)
        elif top == 'models':
            continue
        else:
            _walk(top_path, depth=0)
    return results

if os.path.exists('/kaggle/input'):
    attached = _scan_kaggle_inputs()
    if attached:
        print(f'[PRE-FLIGHT] Discovered {len(attached)} attached manifold(s) in /kaggle/input:')
        for cand in attached:
            print(f'   -> {cand}')
        for cand in attached:
            bname = os.path.basename(cand)
            aliases = {bname, bname.lower()}
            stripped = re.sub(r'^LemGendized', '', bname, flags=re.IGNORECASE)
            if stripped != bname:
                aliases.add(stripped)
                aliases.add(stripped.lower())
                for suf in ('Large', 'Medium', 'Small'):
                    if stripped.endswith(suf):
                        stem = stripped[:-len(suf)]
                        aliases.add(stem)
                        aliases.add(stem.lower())
            for alias in aliases:
                if not alias: continue
                link_name = os.path.join(target_dir, alias)
                if os.path.exists(link_name): continue
                try:
                    os.symlink(cand, link_name)
                    print(f'   -> [OK] [LINKED] {alias} -> {cand}')
                except OSError as e:
                    print(f'   -> [WARN] Symlink failed for {alias}: {e}')
            found.append(cand)

if not found:
    raise RuntimeError(
        f'[ABORT] Nothing attached in /kaggle/input for {model_key}. '
        f'Attach the dataset via the Kaggle sidebar (Add Input -> Your Datasets) '
        f'and re-run from the top.'
    )

print(f'[OK] [DATA] {len(found)} manifold symlink(s) ready in {target_dir}.')


## 6. Checkpoint & Metric Recovery


In [ ]:
import os, shutil
model_key = 'mirnet_exposure'
print(f'[RECOVERY] Deep-searching for {model_key} checkpoints...')
hub_root = '/kaggle/working/LemGendaryModels'
model_hub_dir = os.path.join(hub_root, model_key)
ckpt_hub_dir = os.path.join(model_hub_dir, 'checkpoints')
os.makedirs(ckpt_hub_dir, exist_ok=True)

reg_filename = ''
try:
    import yaml
    yaml_path = '/kaggle/working/lemgendary-training-suite/unified_models_v2.yaml'
    if os.path.exists(yaml_path):
        with open(yaml_path, 'r') as f: reg = yaml.safe_load(f)
        reg_filename = reg.get(model_key, {}).get('filename', '')
except Exception as e: print(f'[REMEDY] An error occurred during environment setup: {e}')

target_slugs = [model_key.lower().replace('_', ''), model_key.lower().replace('_', '-'), reg_filename.lower() if reg_filename else '']
target_slugs = [s for s in target_slugs if s]

found_ckpts = []
try:
    import kagglehub
    kh_dl = kagglehub.model_download('lemtreursi/lemgendary-mirnet-exposure-checkpoints/pytorch/default')
    if kh_dl and os.path.exists(kh_dl):
        print(f'   -> [KAGGLEHUB] Dynamic latest model version resolved: {kh_dl}')
        for root_d, _, files in os.walk(kh_dl):
            for f in files:
                if f.lower().endswith(('.pth', '.pt')) and (any(slug in f.lower() for slug in target_slugs) or 'best' in f.lower() or 'latest' in f.lower() or 'progress' in f.lower()):
                    found_ckpts.append(os.path.join(root_d, f))
except Exception as kh_e: pass

if os.path.exists('/kaggle/input'):
    try:
        queue = ['/kaggle/input']
        depths = {'/kaggle/input': 0}
        while queue:
            curr = queue.pop(0)
            depth = depths[curr]
            if depth > 7: continue
            for item in os.listdir(curr):
                path = os.path.join(curr, item)
                if os.path.isdir(path):
                    item_lower = item.lower()
                    if item_lower in ['datasets', 'images', 'train', 'val', 'test', 'validation', 'dataset']:
                        continue
                    depths[path] = depth + 1
                    queue.append(path)
                    
                    if any(slug in item_lower for slug in target_slugs) or 'checkpoint' in item_lower or 'weights' in item_lower or 'models' in item_lower:
                        try:
                            for f in os.listdir(path):
                                if f.lower().endswith('.pth') and (any(slug in f.lower() for slug in target_slugs) or 'best' in f.lower() or 'latest' in f.lower() or 'progress' in f.lower()):
                                    found_ckpts.append(os.path.join(path, f))
                        except:
                            pass
    except Exception:
        pass

found_ckpts = sorted(list(set(found_ckpts)))
if found_ckpts:
    print(f'   -> [FOUND] {len(found_ckpts)} binaries in Kaggle Manifold.')
    for src in found_ckpts:
        if f'/{model_key}/' not in src.replace('\\', '/') and f'{model_key}' not in os.path.basename(src):
            continue
        if not os.path.exists(src):
            print(f'   -> [WARNING] Source missing (Ghost File/Broken Link): {src}')
            continue
        fname = os.path.basename(src)
        target_f = fname
        if 'latest' in fname.lower(): target_f = f'{model_key}_latest.pth'
        elif 'best' in fname.lower(): target_f = f'{model_key}_best.pth'
        elif 'progress' in fname.lower(): target_f = f'{model_key}_progress.pth'
        
        dst = os.path.join(ckpt_hub_dir, target_f)
        if not os.path.exists(dst) or os.path.getsize(src) > os.path.getsize(dst):
            shutil.copy2(src, dst)
            print(f'   -> [OK] Recovered: {fname} -> {target_f}')
    
    metrics_found = False
    for src in found_ckpts:
        for d in [os.path.dirname(os.path.dirname(src)), os.path.dirname(src)]:
            m_path = os.path.join(d, 'metrics.csv')
            if os.path.exists(m_path):
                try:
                    shutil.copy2(m_path, os.path.join(model_hub_dir, 'metrics.csv'))
                    print(f'[METRICS] Recovered metrics.csv from {os.path.basename(d)}')
                    metrics_found = True; break
                except Exception as e: print(f'[REMEDY] An error occurred during environment setup: {e}')
        if metrics_found: break
else: print('   -> [SKIP] No existing checkpoints found in Kaggle Inputs manifold.')


## 7. Nuclear Training Matrix


In [ ]:
import os, subprocess, sys
suite_candidates = ['/kaggle/working/lemgendary-training-suite', '/kaggle/working/model-training/lemgendary-training-suite', '/kaggle/working']
active_suite_dir = next((p for p in suite_candidates if os.path.exists(os.path.join(p, 'training', 'train.py'))), '/kaggle/working/lemgendary-training-suite')
os.chdir(active_suite_dir)
print(f'[OK] [SUITE] Active working directory set to: {os.getcwd()}')

try:
    current_pid = os.getpid()
    ps_out = subprocess.check_output(['ps', '-ef'], text=True)
    for line in ps_out.split('\n'):
        if 'train.py' in line and str(current_pid) not in line:
            parts = line.split()
            if len(parts) > 1:
                pid = int(parts[1])
                print(f'[JANITOR] Killing stale zombie training process (PID {pid})...')
                subprocess.run(['kill', '-9', str(pid)], capture_output=True)
except Exception:
    pass

print(f'[LAUNCH] [NUCLEAR] Initiating Training Matrix for mirnet_exposure...')
cmd = [sys.executable, '-u', 'training/train.py', '--model', f'{model_key}', '--env', 'kaggle', '--auto_sync']
p = subprocess.Popen(cmd)
try:
    p.wait()
except KeyboardInterrupt:
    print('\n[TERMINATED] Training interrupted by user. Terminating training subprocess safely...')
    try:
        p.terminate()
        p.wait(timeout=5)
    except subprocess.TimeoutExpired:
        p.kill()
    print('[OK] Subprocess successfully killed. VRAM and CPU are clean.')
